# FuquaBot (RAG) — Colab Quickstart

This notebook highlights the *crucial* parts of the project: environment setup, API keys, running the RAG pipeline, and kicking off evaluations.

## 1) Clone / upload the repo
Make sure the project folder (with `Untitled2.py`, `server.py`, `eval.py`, `data/`, etc.) is available in Colab. You can upload it as a zip or pull it from your source location.

In [ ]:
# (Optional) if you zipped the repo and uploaded to Colab files
!unzip -o /content/final_proj.zip -d /content
%cd /content/final proj

## 2) Install dependencies
Installs the core libs plus Gemini client. Adjust if you already have a requirements lock.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q uvicorn google-generativeai

## 3) Set API keys (edit with your real keys)
Do **not** hard-code secrets in the notebook; use environment variables.

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "YOUR_DEEPSEEK_KEY"
os.environ["OPENAI_BASE_URL"] = "https://api.deepseek.com"
os.environ["GOOGLE_API_KEY"] = "YOUR_GEMINI_KEY"
os.environ["ARLIAI_API_KEY"] = "YOUR_ARLIAI_KEY"
# If using HF for embeddings that require auth:
# os.environ["HUGGINGFACE_HUB_TOKEN"] = "YOUR_HF_TOKEN"

## 4) Build the RAG index once (uses `data/` PDFs)
This will (re)build FAISS for the current `EMBED_MODEL`. Set `EMBED_MODEL` to try alternatives.

In [ ]:
import os
os.environ["EMBED_MODEL"] = "all-mpnet-base-v2"  # change to test other embeddings

from Untitled2 import ensure_pipeline
ensure_pipeline()  # builds or loads index

## 5) Ask a quick question via the RAG pipeline
Uses the locally loaded index and your LLM key.

In [ ]:
from Untitled2 import answer_question

question = "What is Team Fuqua?"
answer, sources = answer_question(question)
print("Answer:\n", answer)
print("\nSources:")
for s in sources:
    print(f"- {s['name']} (score={s['score']:.4f})")

## 6) Run evaluations (RAG + baselines)
- RAG over current embedding: use `--mode rag`
- Baseline (DeepSeek/OpenAI-compatible): `--mode baseline`
- Gemini: `--mode gemini`
- ArliAI: `--mode arliai`

You can also use `run_all.sh` on a VM, but in Colab it’s safer to run selectively to avoid timeouts.

In [ ]:
# Example: single RAG eval
!python3 eval.py --qa-file eval_qa.jsonl --mode rag --k 5

# Example: baseline
# !python3 eval.py --qa-file eval_qa.jsonl --mode baseline --baseline-model deepseek-chat

# Example: Gemini
# !python3 eval.py --qa-file eval_qa.jsonl --mode gemini --gemini-model gemini-1.5-flash

# Example: ArliAI (single model, shorter timeout recommended)
# !python3 eval.py --qa-file eval_qa.jsonl --mode arliai \
#   --arliai-models "Gemma-3-27B-it" \
#   --arliai-base-url "https://api.arliai.com/v1" \
#   --arliai-timeout 15

## 7) (Optional) Serve the UI locally
Colab isn’t ideal for hosting the FastAPI UI, but you can run uvicorn and tunnel with ngrok if needed:
```bash
uvicorn server:app --host 0.0.0.0 --port 8000
```
Then use a tunnel (e.g., `ngrok http 8000`) to access `index.html`.

## 8) Project layout (what matters)
- `Untitled2.py`: RAG pipeline (data loading, chunking, embeddings, FAISS, LLM call)
- `server.py`: FastAPI app exposing `/chat` and serving `index.html`
- `index.html`: Frontend UI (chat, sources, copy button)
- `eval.py`: Evaluation harness (RAG/baseline/Gemini/ArliAI)
- `run_all.sh`: Batch runner for RAG across multiple embeddings + baselines
- `data/`: PDFs/TXT knowledge base
- `faiss.index`, `index_meta.json`: Cached vector index and metadata

## 9) RAG flow (core logic)
1. Load PDFs/TXT from `data/`
2. Chunk text (size/overlap configurable)
3. Encode chunks with `SentenceTransformer` (embedding model configurable via `EMBED_MODEL`)
4. Build FAISS index (L2) and cache to disk
5. At query time: embed question, search top-K, build context, call LLM
6. Return answer + structured sources (used by UI as links)

## 10) Frontend (index.html) quick notes
- Hard-coded backend URL: `http://localhost:8000`
- Shows sources separately (answer text omits source mentions)
- Copy button on bot bubbles
- Static prompts are baked in (not dynamic)
- If you host the API elsewhere, update `BACKEND_URL` in the script tag

## 11) Evaluations in one shot (VM/Colab)
- Use `run_all.sh` to sweep RAG over 6 embeddings and run baseline/Gemini/ArliAI once each.
- Set keys first:
```bash
export OPENAI_API_KEY="..."; export OPENAI_BASE_URL="https://api.deepseek.com"
export GOOGLE_API_KEY="..."
export ARLIAI_API_KEY="..."
bash run_all.sh
```
- Results: `eval_qa.<mode>*.results.jsonl` next to `eval_qa.jsonl`

## 12) Key environment variables (cheat sheet)
- `EMBED_MODEL`: e.g., `all-mpnet-base-v2` (controls RAG embeddings)
- `OPENAI_API_KEY`, `OPENAI_BASE_URL`: baseline (DeepSeek/OpenAI-compatible)
- `GOOGLE_API_KEY`: Gemini
- `ARLIAI_API_KEY`: ArliAI
- `TOKENIZERS_PARALLELISM=false`: optional, silences tokenizer fork warnings

## 13) Troubleshooting
- Slow or hanging eval: reduce QA size (e.g., `head -n 3 eval_qa.jsonl > /tmp/qa_small.jsonl`), shorten timeouts, or test with curl first.
- Missing model (HF embeddings): ensure correct repo ID (`intfloat/e5-base-v2`, etc.) and HF auth if gated.
- Frontend “Error contacting backend”: ensure uvicorn is running and `BACKEND_URL` points to it; check browser console for HTTP errors.
- Index stale/missing: delete `faiss.index` and `index_meta.json`, rerun RAG (the pipeline rebuilds automatically).